# Normalize radiography

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar

import sys
sys.path.append('../')
import amglib.readers as rd
import amglib.imageutils as amg

%config InlineBackend.figure_format = 'retina'

## Load the data

In [ ]:
path = 'what/ever/path/to/the/data/' # note the trailing slash

dc  = rd.read_image(path+'dc.fits')[::-1] # the [::-1] flips the image vertically, remove on all if not needed
ob  = rd.read_image(path+'ob.fits')[::-1]
img = rd.read_image(path+'image.fits')[::-1]


In [ ]:
pixel_size = 0.037 # mm

## Prepare slab image

In [ ]:
nimg = amg.normalizeImage(img = nimg,     # the sample image
                          ob  = ob,       # open beam image
                          dc  = dc,       # dark current image 
                          neglog  = True, # Compute -log(I/I_0)
                          doseROI = [50,50,70,70]) # Use ROI=[x0,y0,x1,y1] to compute the dose. doseROI=None skips correction

nimg = amg.morph_spot_clean(nimg) # Run spot cleaning

## Show image with color bar and scale bar
Here we show the image with a grey scale color map. Other options for ```cmap``` would be viridis, jet, hot, cool, plasma, bone, etc

In [ ]:
fig,ax = plt.subplots(1)
s = ax.imshow(nimg, vmin=-0.25, vmax = 2, cmap = 'gray')


scalebar = ScaleBar(pixel_size, 'mm') 
ax.add_artist(scalebar)
plt.colorbar(s,ax=ax)

plt.savefig('radiography.png', dpi=300)

## Show image and histogram

In [ ]:
fig,ax = plt.subplots(1,2, figsize = (10,3.5))

width, height = fig.get_size_inches()

print(f"Figure size: {width} x {height} inches")

ax[0].imshow(nslab, vmin=-0.25, vmax = 2, cmap = 'gray')
ax[1].hist(np.ravel(nslab),bins=512);
fig.tight_layout()

plt.savefig('radiography_histogram.png', dpi=300)


## Show image and intensity profile

In [ ]:
fig,ax = plt.subplots(1,2, figsize = (10,3.5))

width, height = fig.get_size_inches()

print(f"Figure size: {width} x {height} inches");

ax[0].imshow(ncyl, vmin=-0.1, vmax = 1.1, cmap = 'gray')
ypos=1250
dy=50
ax[0].axhline(y=1250,c='red')
ax[1].plot(np.mean(ncyl[(ypos-dy//2):(ypos+dy//2)],axis=0),label='Transmission')
ax[1].plot(-np.log(np.mean(ncyl[(ypos-dy//2):(ypos+dy//2)],axis=0)),label='Optical thickness (-ln(Transmission))')
ax[1].legend(fontsize=7)

fig.tight_layout()

plt.savefig('radiography_profile.png', dpi=300);
